<a href="https://colab.research.google.com/github/amegard64/D-veloppement-d-une-WebApp-Interactive-avec-Streamlit---Lina-Ibouchichene-Alexandre-M-gard/blob/main/Projet_Cloud_1_(PySpark).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projet PySpark

## 1) Initialisation & Chargement des données

In [6]:
from google.colab import files

uploaded = files.upload()

Saving Online_Retail_CSV.csv to Online_Retail_CSV.csv


In [7]:
!ls -lah

total 45M
drwxr-xr-x 1 root root 4.0K Feb 19 09:05 .
drwxr-xr-x 1 root root 4.0K Feb 19 07:39 ..
drwxr-xr-x 4 root root 4.0K Jan 16 14:24 .config
-rw-r--r-- 1 root root  44M Feb 19 09:05 Online_Retail_CSV.csv
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OnlineRetail_BigData_Project") \
    .getOrCreate()

spark


In [11]:
df = spark.read.csv(
    "Online_Retail_CSV.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [12]:
df.printSchema()
df.show(5)
print("Nombre de lignes :", df.count())
print("Nombre de colonnes :", len(df.columns))


root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|01/12/2010 08:26|     2,55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|01/12/2010 08:26|     3,39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|01/12/2010 08:26|     2,75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNI

## 2) Exploration & Prétraitement

### Statistiques descriptives

In [15]:
df.show(5)
df.describe().show()

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|01/12/2010 08:26|     2,55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|01/12/2010 08:26|     3,39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|01/12/2010 08:26|     2,75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|01/12/2010 08:26|     3,39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|01/12/2010 08:26|     3,39|     17850|United Kingdom|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
only showing top 5 rows
+-------+------------------+---

In [17]:
from pyspark.sql.functions import countDistinct

df.select(
    countDistinct("CustomerID").alias("nb_clients_distincts"),
    countDistinct("InvoiceNo").alias("nb_factures_distinctes"),
    countDistinct("StockCode").alias("nb_produits_distincts"),
    countDistinct("Country").alias("nb_pays_distincts")
).show()


+--------------------+----------------------+---------------------+-----------------+
|nb_clients_distincts|nb_factures_distinctes|nb_produits_distincts|nb_pays_distincts|
+--------------------+----------------------+---------------------+-----------------+
|                4372|                 25900|                 4070|               38|
+--------------------+----------------------+---------------------+-----------------+



In [27]:
df.select("Quantity").describe().show()

from pyspark.sql.functions import when, col

df_qty_bins = df.withColumn(
    "qty_bin",
    when(col("Quantity") < 0, "<0 (retours?)")
    .when(col("Quantity") == 0, "0")
    .when(col("Quantity").between(1, 5), "1-5")
    .when(col("Quantity").between(6, 10), "6-10")
    .when(col("Quantity").between(11, 20), "11-20")
    .when(col("Quantity").between(21, 50), "21-50")
    .when(col("Quantity").between(51, 100), "51-100")
    .otherwise(">100")
)

df_qty_bins.groupBy("qty_bin") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

+-------+-----------------+
|summary|         Quantity|
+-------+-----------------+
|  count|           541909|
|   mean| 9.55224954743324|
| stddev|218.0811578502349|
|    min|           -80995|
|    max|            80995|
+-------+-----------------+

+-------------+------+
|qty_bin      |count |
+-------------+------+
|1-5          |317418|
|6-10         |81236 |
|11-20        |75540 |
|21-50        |44773 |
|<0 (retours?)|10624 |
|51-100       |7368  |
|>100         |4950  |
+-------------+------+



### Valeurs manquantes

In [20]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show(truncate=False)


+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|0        |0        |1454       |0       |0          |0        |135080    |0      |
+---------+---------+-----------+--------+-----------+---------+----------+-------+



L’analyse des valeurs manquantes montre que la variable `CustomerID` contient un nombre significatif d’observations nulles.  
Ces transactions ne peuvent pas être associées à un client et ne sont donc pas exploitables dans le cadre d’une segmentation client ou d’une modélisation comportementale.

Par conséquent, les lignes présentant une valeur manquante sur `CustomerID` seront supprimées.  
Concernant la variable `Description`, les valeurs manquantes n’impactent pas les analyses prévues et pourront être conservées ou ignorées sans traitement spécifique.

Aucune imputation n’est réalisée à ce stade afin d’éviter l’introduction de biais artificiels dans les variables clés utilisées pour la suite du projet.


Suppression des lignes présentant une valeur manquante sur `CustomerID`:

In [21]:
from pyspark.sql.functions import col

df_clean = df.filter(col("CustomerID").isNotNull())

print("Nombre de lignes avant filtrage :", df.count())
print("Nombre de lignes après filtrage :", df_clean.count())


Nombre de lignes avant filtrage : 541909
Nombre de lignes après filtrage : 406829


### Valeurs anormales

Lors de la phase d’initialisation et de chargement des données, nous avons observé que la variable `UnitPrice` était encodée sous forme de chaîne de caractères (`string`).  
Ce format est dû à l’utilisation d’un séparateur décimal de type européen (virgule), qui empêche toute comparaison numérique directe.

Avant de procéder à la détection des valeurs anormales, il est donc nécessaire de convertir cette variable en format numérique standard afin de garantir la validité des traitements et des analyses statistiques ultérieures.

In [28]:
from pyspark.sql.functions import regexp_replace, col

df_clean = df_clean.withColumn(
    "UnitPrice",
    regexp_replace(col("UnitPrice"), ",", ".").cast("double")
)


In [29]:
from pyspark.sql.functions import col, sum

df_clean.select(
    sum((col("Quantity") <= 0).cast("int")).alias("quantity_le_0"),
    sum((col("UnitPrice") <= 0).cast("int")).alias("unitprice_le_0")
).show()


+-------------+--------------+
|quantity_le_0|unitprice_le_0|
+-------------+--------------+
|         8905|            40|
+-------------+--------------+



La présence de quantités ou de prix unitaires négatifs ou nuls est observée dans une partie du jeu de données.  
Même si la nature exacte de ces observations n’est pas documentée, elles ne correspondent pas à des transactions d’achat positives exploitables dans le cadre d’une analyse du comportement client.

Afin de garantir la cohérence du calcul des indicateurs RFM et l’interprétabilité des résultats de segmentation et de modélisation, ces transactions sont exclues du périmètre de l’étude.

In [30]:
df_clean = df_clean.filter(
    (col("Quantity") > 0) & (col("UnitPrice") > 0)
)

print("Nombre de lignes après suppression des valeurs anormales :", df_clean.count())


Nombre de lignes après suppression des valeurs anormales : 397884


### Conversion de (`InvoiceDate`) en Timestamp

La variable `InvoiceDate` est initialement encodée sous forme de chaîne de caractères.  
Afin de permettre les analyses temporelles (notamment le calcul de la récence dans la construction des variables RFM), cette variable est convertie en format `Timestamp`.


In [31]:
from pyspark.sql.functions import to_timestamp

df_clean = df_clean.withColumn(
    "InvoiceDate",
    to_timestamp("InvoiceDate", "dd/MM/yyyy HH:mm")
)

df_clean.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



### Création de la variable (`TotalPrice`)

Afin de faciliter le calcul des indicateurs de type Monetary dans la construction des variables RFM, une variable de dépense totale par ligne de transaction est créée.  
Cette variable correspond au produit de la quantité achetée et du prix unitaire.


In [32]:
from pyspark.sql.functions import col

df_clean = df_clean.withColumn(
    "TotalPrice",
    col("Quantity") * col("UnitPrice")
)

In [33]:
df_clean.select("Quantity", "UnitPrice", "TotalPrice").show(5)

+--------+---------+------------------+
|Quantity|UnitPrice|        TotalPrice|
+--------+---------+------------------+
|       6|     2.55|15.299999999999999|
|       6|     3.39|             20.34|
|       8|     2.75|              22.0|
|       6|     3.39|             20.34|
|       6|     3.39|             20.34|
+--------+---------+------------------+
only showing top 5 rows
